# 30장 · 검색 함수를 필수 Retriever Node로 연결하기

검색 문장을 화면에 그대로 띄우면 어느 쪽의 안내인지 찾기 어렵습니다. 26장에서 문자열로 담았던 검색 결과에 문서 ID, 원문 제목, 쪽수를 함께 싣고 같은 검색 함수를 그래프 안에서 호출합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langgraph 1.2.16입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 문장 하나를 출처가 있는 검색 결과로 받기

노트북의 자료는 앞서 사용한 가상 도서관 안내 세 쪽입니다. 읽는 법에 집중하려고 로컬 검색 함수는 세 낱말의 유무를 거리로 바꾸며 k개를 고릅니다. Chroma 서버나 외부 임베딩을 호출하는 셀은 없습니다.



**볼 곳**　‘검색 문장’와 ‘출처와 쪽’가 오른쪽의 ‘검색 레코드’에 함께 들어가는 위치를 보세요.

**애니메이션의 의미**　검색 문장에 문서 ID와 원문 위치를 함께 붙입니다.


### 셀 01 · 검색된 한 쪽 펼치기

대출 질문을 넣고 목록의 첫 번째 레코드를 읽으세요.

In [1]:
records = [{'id': 'library-1',
  'text': '도서 대출 기간은 14일입니다.',
  'source': '가상 도서관 이용 안내',
  'page': 1},
 {'id': 'library-2',
  'text': '도서관은 월요일에 휴관합니다.',
  'source': '가상 도서관 이용 안내',
  'page': 2},
 {'id': 'library-3',
  'text': '무인 반납함은 24시간 이용할 수 있습니다.',
  'source': '가상 도서관 이용 안내',
  'page': 3}]
terms = ("대출", "휴관", "반납")

def retrieve(question, k=1):
    query = [int(term in question) for term in terms]
    if not any(query):
        return []
    def distance(row):
        vector = [int(term in row["text"]) for term in terms]
        return sum((a-b)**2 for a, b in zip(query, vector)), row["id"]
    return sorted(records, key=distance)[:k]

print(retrieve('대출 기간은?', 1)[0])

{'id': 'library-1', 'text': '도서 대출 기간은 14일입니다.', 'source': '가상 도서관 이용 안내', 'page': 1}


library-1에 14일이라는 문장, 가상 도서관 이용 안내, 1쪽이 함께 있습니다. text만 남기면 다음 노드에서 출처를 복원할 수 없습니다.

## 답변 전에 검색을 한 번 거치기

검색 결과를 상태에 넣을 때는 documents 키에 목록을 그대로 반환합니다. 다음 노드는 같은 목록에서 답을 꺼냅니다. 별도로 다시 검색하면 처음 확인한 근거와 다른 문서를 답에 쓰게 될 수 있습니다.



**볼 곳**　‘질문’에서 출발해 ‘필수 검색 노드’를 거친 뒤 ‘근거로 답변’가 되는 순서를 따라가세요.

**애니메이션의 의미**　답변 노드 앞에 검색 노드를 두어 매번 근거를 찾습니다.


### 셀 02 · 검색과 답변 노드 연결하기

아래 그래프에서 START 다음에 연결한 노드를 찾고 휴관 질문을 실행하세요.

이 노트북의 retrieve는 공백 질문에서 []를 반환하고 답변은 상태의 answer에서 읽습니다. 뒤의 R1/A1 참고 구현은 공백을 ValueError로 처리하며 result 안에 답을 넣습니다. 두 구현의 키를 섞으면 값을 찾을 수 없습니다.

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
class State(TypedDict):
    question: str
    documents: list[dict]
    answer: str

def search_node(state):
    return {"documents": retrieve(state["question"], k=1)}

def answer_node(state):
    docs = state["documents"]
    return {"answer": docs[0]["text"] if docs else "제공된 자료에서 확인할 수 없습니다."}

builder = StateGraph(State)
builder.add_node("search", search_node)
builder.add_node("compose", answer_node)
builder.add_edge(START, "search")
builder.add_edge("search", "compose")
builder.add_edge("compose", END)
graph = builder.compile()

request = {'question': '휴관 요일은 언제인가요?', 'documents': [], 'answer': ''}
result = graph.invoke(request)
print(result['answer'], result['documents'][0]['page'])

도서관은 월요일에 휴관합니다. 2


월요일이라는 답 뒤에 2가 붙습니다. 검색 함수가 고른 2쪽을 compose가 그대로 읽었습니다. 답 옆의 숫자는 결과 순위가 아닌 원문의 쪽수입니다. result는 실행 결과 전체의 변수 이름입니다. 답은 result["answer"], 원문 쪽수는 result["documents"][0]["page"]에서 읽습니다.

### 과제 03 · 문서를 찾지 못한 질문 처리하기

‘주차 요금은?’을 넣고 documents와 answer를 차례로 확인하세요.

In [3]:
# 초기 상태의 질문을 바꾸어 그래프를 실행하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>empty = graph.invoke({**request, &#x27;question&#x27;: &#x27;주차 요금은?&#x27;})
print(empty[&#x27;documents&#x27;], empty[&#x27;answer&#x27;])</code></pre>
<p>실행 결과</p><pre><code>[] 제공된 자료에서 확인할 수 없습니다.</code></pre>
<p>빈 목록과 확인할 수 없다는 안내가 나옵니다. 세 낱말 모두 질문에 없어 검색 함수가 빈 목록을 돌려준 경우입니다. 연결 오류에 같은 안내를 쓰면 장애가 가려집니다.</p>
</details>

## 돌아보기

Chroma에서 같은 결과를 받을 때도 id, text, source, page 네 칸이 필요합니다. 여러 열로 나뉜 반환값을 합칠 때 목록 길이와 쪽수 단위를 확인하세요.